# Task 3: Re-check Before the Board

Task 2 answered the leadership team's questions on the cleaned data. Before those answers reach the board, this notebook tests the yardstick behind them:

1. **Is the extract complete?** Which company codes report in each month.
2. **Can the margin be trusted?** Whether recorded costs are real, and whether the company's own entities are mixed in with customers.
3. **What do the six findings look like** on external customers, using only lines with a real cost?

Every figure and chart on the [case study page](https://racheltran184.github.io/Python-Project-2/) comes from this notebook; the last cell writes them to `briefing_figures.json`.

**Input:** `dataset_cleaned.csv` from Task 1 (1,087,121 lines, AUD), plus the raw `Dataset1.csv` and `Dataset2.csv` for one date check. Uses pandas and NumPy only.

In [1]:
import json
import numpy as np
import pandas as pd

# Read codes as text so identifiers keep their exact form
CODE_COLS = ['company_code', 'customer_code', 'customer_district_code', 'item_code',
             'order_type_code', 'invoice_number', 'light_source']
df = pd.read_csv('dataset_cleaned.csv', dtype={c: str for c in CODE_COLS}, parse_dates=['accounting_date'], low_memory=False)

S, C, Q = 'value_sales_aud', 'value_cost_aud', 'value_quantity'
YEARS = df['accounting_year'].nunique()  # 2020 to 2024

# Districts 710 (head office), 720 (intercompany) and 545 (head office NZ) are the company's own entities
INTERNAL_DISTRICTS = ['710', '720', '545']
df['internal'] = df['customer_district_code'].isin(INTERNAL_DISTRICTS)

# A recorded cost equal to the quantity sold is a $1.00 unit cost: a placeholder, not a real cost
df['placeholder_cost'] = (df['value_cost'] == df[Q]) & (df[Q] != 0)

def gross_margin(x):
    """Gross margin %: (sales - cost) / sales."""
    return 100 * (x[S].sum() - x[C].sum()) / x[S].sum()

ext = df[~df['internal']]                   # external customers
ext_costed = ext[~ext['placeholder_cost']]  # external customers, lines with a real cost

print(f"{len(df):,} lines, {df['accounting_date'].min():%b %Y} to {df['accounting_date'].max():%b %Y} ({YEARS} years)")
print(f"External lines: {len(ext):,}; of these, lines with a real cost: {len(ext_costed):,}")

1,087,121 lines, Jan 2020 to Dec 2024 (5 years)
External lines: 1,027,141; of these, lines with a real cost: 496,919


## 1. Is the extract complete?

Task 2's monthly chart showed sales falling 99% from May 2020 to December 2024. Before reading that as a collapse, check which company codes report in each month.

In [2]:
monthly = df.groupby(df['accounting_date'].dt.to_period('M')).agg(
    sales_m=(S, lambda s: s.sum() / 1e6), company_codes=('company_code', 'nunique'))
annual = df.groupby('accounting_year')[S].sum() / 1e6
first_half = df[df['accounting_month'] <= 6].groupby('accounting_year')[S].sum() / 1e6
last_month = df.groupby('company_code')['accounting_month'].max()

print('Annual sales, AUD M:      ', annual.round(1).to_dict())
print('January-June share, %:    ', (100 * first_half / annual).round(1).to_dict())
print(f"All years, January-June:   {100 * first_half.sum() / annual.sum():.1f}% of sales")
print(f"Company codes with no July-December lines in any year: {(last_month <= 6).sum()} of {len(last_month)}")
print(f"May 2020 ${monthly.loc['2020-05', 'sales_m']:.1f}M, May 2024 ${monthly.loc['2024-05', 'sales_m']:.1f}M, "
      f"December 2024 ${monthly.loc['2024-12', 'sales_m']:.1f}M")
print('\nCompany codes reporting, by calendar month (fewest and most across the five years):')
print(monthly.groupby(monthly.index.month)['company_codes'].agg(['min', 'max']).T.to_string())

Annual sales, AUD M:       {2020: 256.6, 2021: 249.3, 2022: 243.5, 2023: 244.0, 2024: 246.0}
January-June share, %:     {2020: 98.0, 2021: 98.0, 2022: 97.9, 2023: 98.2, 2024: 98.0}
All years, January-June:   98.0% of sales
Company codes with no July-December lines in any year: 7 of 10
May 2020 $48.1M, May 2024 $47.1M, December 2024 $0.4M

Company codes reporting, by calendar month (fewest and most across the five years):
accounting_date  1   2   3   4   5   6   7   8   9   10  11  12
min               8   9   9   9   9   9   3   3   3   3   3   3
max               9  10  10   9   9  10   3   3   3   3   3   3


In [3]:
# Task 1 drops lines whose date can't be read. Count them in the raw exports.
raw = pd.concat([pd.read_csv(f, usecols=['accounting_date', 'value_sales'], encoding='ISO-8859-1')
                 for f in ['Dataset1.csv', 'Dataset2.csv']], ignore_index=True)
as_text = pd.to_numeric(raw['accounting_date'], errors='coerce').fillna(0).astype(int).astype(str)
impossible = raw['accounting_date'].notna() & pd.to_datetime(as_text, format='%Y%m%d', errors='coerce').isna()

print(f"Raw lines:                      {len(raw):,}")
print(f"No date:                        {raw['accounting_date'].isna().sum():,}")
print(f"Impossible dates:               {impossible.sum():,} ({', '.join(sorted(as_text[impossible].unique()))}), "
      f"about ${raw.loc[impossible, 'value_sales'].sum() / 1e6:.1f}M of sales")
other = len(raw) - raw['accounting_date'].isna().sum() - impossible.sum() - len(df)
print(f"Other lines removed in Task 1:  {other} (6 mostly empty, 43 with invalid codes; see Task 1's log), leaving {len(df):,}")

Raw lines:                      1,089,726
No date:                        4
Impossible dates:               2,552 (20210229, 20220229, 20230229), about $3.1M of sales
Other lines removed in Task 1:  49 (6 mostly empty, 43 with invalid codes; see Task 1's log), leaving 1,087,121


**Read:** annual sales are flat, but every year 98% of them fall in January to June, because seven of the ten company codes never report July to December. The "99% fall" compares a full month with one where only three codes report. The 2,552 lines dated 29 February in non-leap years are dropped by Task 1; they hold about $1M a year and should be re-dated rather than dropped in the next extract.

## 2. Can the margin be trusted?

In [4]:
print(f"Lines where the recorded cost equals the quantity ($1.00 a unit): {100 * df['placeholder_cost'].mean():.1f}%")

# Are these genuine $1.00 products? Check whether the same products carry other costs on other lines.
by_product = df.groupby('item_code').agg(lines=('placeholder_cost', 'size'), placeholder=('placeholder_cost', 'sum'))
regular = by_product[(by_product['lines'] >= 10) & (by_product['placeholder'] > 0)]
print(f"Products sold on 10+ lines that have placeholder costs: {len(regular):,}; "
      f"{100 * (regular['placeholder'] < regular['lines']).mean():.1f}% also carry a real cost on other lines")

sold = ext[(ext[Q] > 0) & (ext[S] > 0)]
top_item = sold.groupby('item_code')['invoice_number'].nunique().idxmax()
t = sold[sold['item_code'] == top_item]
tc = t[~t['placeholder_cost']]
print(f"Most-ordered product ({top_item}): real unit cost ${tc[C].sum() / tc[Q].sum():.2f}, "
      f"unit price ${t[S].sum() / t[Q].sum():.2f}")

margins = {
    'Raw file, every line': gross_margin(df),
    'Lines with a recorded cost': gross_margin(df[~df['placeholder_cost']]),
    'External customers': gross_margin(ext_costed),
    'Intercompany and head office': gross_margin(df[df['internal'] & ~df['placeholder_cost']]),
}
print('\nGross margin, %')
for k, v in margins.items():
    print(f"  {k:<30}{v:5.1f}")
print(f"\nIntercompany and head office: {100 * df.loc[df['internal'], S].sum() / df[S].sum():.1f}% of sales")

Lines where the recorded cost equals the quantity ($1.00 a unit): 51.6%
Products sold on 10+ lines that have placeholder costs: 6,727; 96.8% also carry a real cost on other lines
Most-ordered product (25550): real unit cost $3.05, unit price $5.68

Gross margin, %
  Raw file, every line           65.9
  Lines with a recorded cost     36.1
  External customers             42.6
  Intercompany and head office   12.8

Intercompany and head office: 22.2% of sales


**Read:** half the lines carry a $1.00 placeholder cost, and nearly every affected product has a real cost elsewhere, so the 66% margin in the raw file isn't real. A fifth of sales go to the company's own entities at a thin margin. The yardstick for everything below is **external customers, on lines with a real cost**.

## 3. The six findings
External customers only; margins on lines with a real cost; "a year" is the 2020-2024 average.

### Finding 1: sales are steady, but the mix isn't moving

In [5]:
FAMILY = {'TRADITIONAL': 'Traditional', 'ACCESSORIES': 'Accessories', 'LED': 'LED'}
family = ext['light_source'].map(FAMILY)
mix = ext.groupby(['accounting_year', family])[S].sum().unstack() / 1e6
print('External sales by product family, % of each year')
print((100 * mix.div(mix.sum(axis=1), axis=0)).round(1).to_string())
print('\nExternal sales, AUD M:', mix.sum(axis=1).round(1).to_dict())
print('LED sales, AUD M:     ', mix['LED'].round(1).to_dict())
active = ext[ext[S] > 0].groupby('accounting_year')['customer_code'].nunique()
print('Active external customers each year:', active.to_dict())

External sales by product family, % of each year
light_source     Accessories   LED  Traditional
accounting_year                                
2020                    18.5  12.1         69.4
2021                    17.4  11.7         70.9
2022                    18.3  11.9         69.9
2023                    17.9  12.3         69.8
2024                    17.6  11.3         71.1

External sales, AUD M: {2020: 196.9, 2021: 196.5, 2022: 187.9, 2023: 191.2, 2024: 192.2}
LED sales, AUD M:      {2020: 23.8, 2021: 23.0, 2022: 22.3, 2023: 23.5, 2024: 21.7}
Active external customers each year: {2020: 3445, 2021: 3434, 2022: 3442, 2023: 3451, 2024: 3441}


### Finding 2: margin is made in the mix and the channel, not in projects

In [6]:
print('Gross margin by product family (share of external sales)')
fam_sales = ext.groupby(family)[S].sum()
for f in ['Accessories', 'Traditional', 'LED']:
    g = ext_costed[ext_costed['light_source'].map(FAMILY) == f]
    print(f"  {f:<14}{gross_margin(g):5.1f}%   ({100 * fam_sales[f] / fam_sales.sum():.1f}% of sales)")

CHANNELS = {'EDI': 'EDI (electronic reorder)', 'NOR': 'Normal order', 'PRO': 'Project order', 'PRD': 'Project, site delivery'}
print('\nGross margin by order channel')
for code_, name in CHANNELS.items():
    sales = ext.loc[ext['order_type_code'] == code_, S].sum()
    gm = gross_margin(ext_costed[ext_costed['order_type_code'] == code_])
    print(f"  {name:<26}{gm:5.1f}%   ${sales / 1e6:6.1f}M over five years ({100 * sales / ext[S].sum():.1f}% of sales)")

Gross margin by product family (share of external sales)
  Accessories    44.9%   (17.9% of sales)
  Traditional    42.6%   (70.2% of sales)
  LED            40.0%   (11.8% of sales)

Gross margin by order channel
  EDI (electronic reorder)   45.1%   $  40.8M over five years (4.2% of sales)
  Normal order               44.1%   $ 665.5M over five years (69.0% of sales)
  Project order              43.8%   $  90.6M over five years (9.4% of sales)
  Project, site delivery     42.7%   $ 103.2M over five years (10.7% of sales)


### Finding 3: a fifth of customers bring most of the sales
Customers are ranked by orders a year (invoices with a sale), then split into five equal groups.

In [7]:
cust = ext[ext[S] > 0].groupby('customer_code').agg(gross=(S, 'sum'), orders=('invoice_number', 'nunique'))
cust['orders_year'] = cust['orders'] / YEARS
cust['spend_year'] = cust['gross'] / YEARS
cust['fifth'] = pd.qcut(cust['orders_year'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

fifths = cust.groupby('fifth').agg(customers=('gross', 'size'), median_orders_year=('orders_year', 'median'),
                                   median_spend_year=('spend_year', 'median'), sales=('gross', 'sum'))
fifths['share_of_sales'] = 100 * fifths['sales'] / cust['gross'].sum()
print(f"External customers with sales: {len(cust):,}")
print(fifths.drop(columns='sales').round(1).to_string())

top10 = cust['gross'].nlargest(int(round(0.1 * len(cust)))).sum() / cust['gross'].sum()
x, y = np.log(cust['orders_year']), np.log(cust['spend_year'])
slope = np.polyfit(x, y, 1)[0]
r2 = np.corrcoef(x, y)[0, 1] ** 2
top2 = cust[cust['fifth'] >= 4]
print(f"\nTop 10% of customers by sales: {100 * top10:.1f}% of sales")
print(f"Spend against orders (log-log): slope {slope:.2f}, R-squared {r2:.2f}")
print(f"Two most frequent fifths: {len(top2):,} accounts, {top2['orders_year'].sum():,.0f} orders and "
      f"${top2['spend_year'].sum() / 1e6:.1f}M a year, ${top2['gross'].sum() / top2['orders'].sum():,.0f} an order")
print(f"10% more orders from them at that value: ${0.1 * top2['spend_year'].sum() / 1e6:.1f}M a year (illustration)")

External customers with sales: 3,884
       customers  median_orders_year  median_spend_year  share_of_sales
fifth                                                                  
1            777                 0.4              372.3             1.3
2            777                 1.8             1611.2             3.5
3            776                 6.4             6135.2             6.7
4            777                16.8            24251.5            15.6
5            777                57.6           115808.1            72.9

Top 10% of customers by sales: 64.3% of sales
Spend against orders (log-log): slope 1.04, R-squared 0.72
Two most frequent fifths: 1,554 accounts, 67,459 orders and $177.9M a year, $2,637 an order
10% more orders from them at that value: $17.8M a year (illustration)


### Finding 4: volume is sold at commodity prices
Products are ranked by units sold, then split into ten equal groups.

In [8]:
prod = sold.groupby('item_code').agg(units=(Q, 'sum'), sales=(S, 'sum'))
prod['unit_price'] = prod['sales'] / prod['units']
prod['tenth'] = pd.qcut(prod['units'].rank(method='first'), 10, labels=list(range(1, 11))).astype(int)
tenths = prod.groupby('tenth').agg(products=('units', 'size'), median_unit_price=('unit_price', 'median'), sales=('sales', 'sum'))
tenths['share_of_sales'] = 100 * tenths['sales'] / prod['sales'].sum()
print(f"Products sold: {len(prod):,}")
print(tenths.drop(columns='sales').round(2).to_string())
price_slope = np.polyfit(np.log(prod['units']), np.log(prod['unit_price']), 1)[0]
print(f"\nEach doubling of volume comes with a unit price {100 * (1 - 2 ** price_slope):.0f}% lower (log-log slope {price_slope:.2f})")

Products sold: 21,577
       products  median_unit_price  share_of_sales
tenth                                             
1          2158             280.00            0.60
2          2158             211.32            0.65
3          2157             195.00            1.02
4          2158             192.79            1.56
5          2158             161.52            2.47
6          2157             137.54            3.71
7          2158             101.72            5.91
8          2157              64.73           10.07
9          2158              24.77           17.25
10         2158               6.34           56.76

Each doubling of volume comes with a unit price 26% lower (log-log slope -0.42)


### Finding 5: credits flow back every year
A credit is any line with negative sales; the credit rate compares credits with gross (positive) sales.

In [9]:
credit = ext[ext[S] < 0]
gross_sales = ext.loc[ext[S] > 0, S].sum()
CREDIT_TYPES = {'CRR': 'Goods returned', 'CRD': 'Price-only credit', 'CPR': 'Project goods returned'}
credit_type = credit['order_type_code'].map(CREDIT_TYPES).fillna('Other credit types')
by_type = -credit.groupby(credit_type)[S].sum() / YEARS / 1e6
print(f"Credits: ${-credit[S].sum() / YEARS / 1e6:.2f}M a year, {100 * -credit[S].sum() / gross_sales:.1f}% of gross external sales")
print(by_type.sort_values(ascending=False).round(2).to_string())

district_credit = (100 * -credit.groupby('customer_district_code')[S].sum()
                   / ext[ext[S] > 0].groupby('customer_district_code')[S].sum())

Credits: $7.98M a year, 4.0% of gross external sales
order_type_code
Goods returned            4.01
Price-only credit         2.75
Other credit types        0.78
Project goods returned    0.44


### Finding 6: regions are closer than they look

In [10]:
DISTRICTS = {'200': 'Sydney', '300': 'Melbourne', '400': 'Brisbane', '600': 'Perth', '500': 'Adelaide',
             '410': 'Townsville', '210': 'ACT/Riverina', '540': 'Northern Region NZ', '510': 'Darwin',
             '310': 'Tasmania', '530': 'South Island NZ', '535': 'Central Region NZ', '520': 'NZ specialist brand'}
dist = pd.DataFrame({
    'sales_m_year': ext.groupby('customer_district_code')[S].sum() / YEARS / 1e6,
    'gross_margin': pd.Series({d: gross_margin(g) for d, g in ext_costed.groupby('customer_district_code')}),
    'credit_rate': district_credit,
}).sort_values('sales_m_year', ascending=False)
print(dist.rename(index=DISTRICTS).round(1).to_string())

LARGEST_FIVE, SMALLER_AU = ['200', '300', '400', '600', '500'], ['410', '210', '510', '310']
gm_five = gross_margin(ext_costed[ext_costed['customer_district_code'].isin(LARGEST_FIVE)])
gm_four = gross_margin(ext_costed[ext_costed['customer_district_code'].isin(SMALLER_AU)])
print(f"\nFive largest districts {gm_five:.1f}%; four smaller Australian districts {gm_four:.1f}%")

                     sales_m_year  gross_margin  credit_rate
Sydney                       42.0          43.0          3.7
Melbourne                    39.1          41.3          4.7
Brisbane                     35.2          42.2          3.6
Perth                        30.8          42.3          4.2
Adelaide                     15.9          41.8          2.8
Townsville                    7.4          45.3          2.9
ACT/Riverina                  6.7          45.8          2.7
Northern Region NZ            5.4          40.0          8.8
Darwin                        3.4          45.2          2.6
Tasmania                      2.9          43.0          3.3
South Island NZ               1.8          46.0          3.9
Central Region NZ             1.3          46.7          6.0
NZ specialist brand           1.1          54.5          3.0

Five largest districts 42.2%; four smaller Australian districts 45.2%


## 4. What held up

| Question | Task 2 result | Re-checked here | Verdict |
|---|---|---|---|
| Is the business shrinking? | 99% fall from the 2020 peak | Flat at \$244M to \$257M a year; 7 of 10 company codes never report July to December | Changed |
| Gross margin | 62% average | 42.6% to external customers, on lines with a real cost | Changed |
| Projects against normal orders (Welch t-test) | Projects +13.7 points | 43.8% against 44.1%: no premium | Changed |
| Metro against regional districts (Welch t-test) | 0.6 points apart, negligible | Regional 45.2%, metro 42.2% | Sharper |
| LED against Traditional (Welch t-test) | LED 2.8 points lower | LED 40.0%, Traditional 42.6% | Holds |
| What drives customer value? (OLS) | Order frequency, R² 0.81 | Spend rises about one-for-one with orders (R² 0.72 on orders alone) | Holds |
| What drives unit price? (OLS) | Volume is the strongest discount, R² 0.56 | Highest-volume tenth sells at \$6.34 a unit, lowest-volume tenth at \$280 | Holds |

## 5. Figures for the case study
The page's charts are drawn from this file.

In [11]:
r1, r2_ = lambda v: round(float(v), 1), lambda v: round(float(v), 2)
figures = {
    'monthly': [{'ym': str(p), 'v': r2_(row.sales_m), 'units': int(row.company_codes)} for p, row in monthly.iterrows()],
    'units_total': int(df['company_code'].nunique()),
    'years': [{'y': int(y), 'v': r1(annual[y]), 'h1': r1(first_half[y])} for y in annual.index],
    'h1_share': r1(100 * first_half.sum() / annual.sum()),
    'margins': [{'k': k, 'v': r1(v), 'role': role} for (k, v), role in zip(margins.items(), ['ctx', 'mid', 'ink', 'ctx'])],
    'placeholder_rows': r1(100 * df['placeholder_cost'].mean()),
    'ic_share': r1(100 * df.loc[df['internal'], S].sum() / df[S].sum()),
    'mix': [{'y': int(y), 'trad': r1(100 * mix.loc[y, 'Traditional'] / mix.loc[y].sum()),
             'acc': r1(100 * mix.loc[y, 'Accessories'] / mix.loc[y].sum()), 'led': r1(100 * mix.loc[y, 'LED'] / mix.loc[y].sum()),
             'tot': r1(mix.loc[y].sum()), 'led_v': r1(mix.loc[y, 'LED'])} for y in mix.index],
    'light_gm': [{'k': f, 'v': r1(gross_margin(ext_costed[ext_costed['light_source'].map(FAMILY) == f])),
                  'share': r1(100 * fam_sales[f] / fam_sales.sum())} for f in ['Accessories', 'Traditional', 'LED']],
    'channels': [{'k': name, 'code': c, 'v': r1(gross_margin(ext_costed[ext_costed['order_type_code'] == c])),
                  'sales': r1(ext.loc[ext['order_type_code'] == c, S].sum() / 1e6),
                  'share': r1(100 * ext.loc[ext['order_type_code'] == c, S].sum() / ext[S].sum())} for c, name in CHANNELS.items()],
    'freq': [{'q': int(q), 'n': int(row.customers), 'share': r1(row.share_of_sales), 'med_year': int(round(row.median_spend_year)),
              'inv_year': r1(row.median_orders_year)} for q, row in fifths.iterrows()],
    'customers_ext': int(len(cust)), 'top10_share': r1(100 * top10), 'freq_beta': r2_(slope), 'freq_r2': r2_(r2),
    'top2_inv_year': int(round(top2['orders_year'].sum())), 'top2_sales_year': r1(top2['spend_year'].sum() / 1e6),
    'top2_per_invoice': int(round(top2['gross'].sum() / top2['orders'].sum())),
    'price': [{'d': int(d), 'p': r2_(row.median_unit_price), 'share': r1(row.share_of_sales)} for d, row in tenths.iterrows()],
    'price_beta': r2_(price_slope),
    'credits': [{'k': k, 'code': c, 'v': r2_(by_type.get(k, 0))} for c, k in
                [('CRR', 'Goods returned'), ('CRD', 'Price-only credit'), ('OTHER', 'Other credit types'), ('CPR', 'Project goods returned')]],
    'credits_year': r1(-credit[S].sum() / YEARS / 1e6), 'credit_rate': r1(100 * -credit[S].sum() / gross_sales),
    'price_only_year': r2_(by_type['Price-only credit']),
    'districts': [{'k': DISTRICTS[d], 'code': d, 'sales': r1(row.sales_m_year), 'gm': r1(row.gross_margin), 'credit': r1(row.credit_rate)}
                  for d, row in dist.iterrows()],
    'gm_metro': r1(gm_five), 'gm_regional': r1(gm_four),
    'ext_sales_year': r1(ext[S].sum() / YEARS / 1e6), 'sales_year': r1(df[S].sum() / YEARS / 1e6),
    'credits_year_exact': r2_(-credit[S].sum() / YEARS / 1e6),
}
with open('briefing_figures.json', 'w') as f:
    json.dump(figures, f, separators=(',', ':'))
print(f"Wrote briefing_figures.json ({len(figures)} figure groups)")

Wrote briefing_figures.json (30 figure groups)
